# L1 · LangChain paso a paso con un mensaje de cliente de ACME

*Advanced Topics in Analytics II - Generative AI* · Instructor: Sergio A. Mora Pardo

Seguimos el **capítulo 1 de Learning LangChain**, de Mayo Oshin y Nuno Campos, con el caso que acompaña todo el curso: **ACME**, una fintech ficticia. Un cliente escribe al soporte de ACME por un cargo duplicado en su tarjeta. El caso es pequeño: leer el mensaje y responder preguntas sobre lo que dice. No consulta cuentas ni resuelve el reclamo; eso llegará con las herramientas y los agentes.

## Objetivos

Al terminar podrás:

1. Llamar a Claude desde LangChain con texto y con mensajes de sistema y de usuario.
2. Separar instrucciones y datos con plantillas reutilizables.
3. Pedir salidas con forma: un objeto estructurado o una lista.
4. Elegir entre `invoke`, `batch` y `stream` según el caso.
5. Conectar pasos en una cadena con `@chain` y con `|`, y añadirle pasos nuevos.

## Metodología

- **Un concepto por sección.** Cada sección añade una sola idea sobre la anterior.
- **Leer, ejecutar, observar.** Lee la explicación, ejecuta la celda y observa la salida antes de avanzar. Las salidas del modelo pueden variar.
- **Mismo mensaje y mismo modelo** en todo el recorrido, para que solo cambie el concepto. Los imports aparecen cuando se necesitan.
- **Práctica al final:** cambiarás el mensaje y las preguntas por otro caso de ACME.

## Contenido

- [0. Preparación del modelo](#0.-Preparación-del-modelo)
- [1. Enviar un texto al modelo](#1.-Enviar-un-texto-al-modelo)
- [2. Enviar un mensaje de usuario](#2.-Enviar-un-mensaje-de-usuario)
- [3. Añadir instrucciones de sistema](#3.-Añadir-instrucciones-de-sistema)
- [4. Construir una plantilla de texto](#4.-Construir-una-plantilla-de-texto)
- [5. Enviar la plantilla al modelo](#5.-Enviar-la-plantilla-al-modelo)
- [6. Construir una plantilla de mensajes](#6.-Construir-una-plantilla-de-mensajes)
- [7. Enviar los mensajes al modelo](#7.-Enviar-los-mensajes-al-modelo)
- [8. Pedir una salida estructurada](#8.-Pedir-una-salida-estructurada)
    - [8.1 Añadir un campo](#8.1-Añadir-un-campo)
- [9. Convertir texto separado por comas en una lista](#9.-Convertir-texto-separado-por-comas-en-una-lista)
- [10. Elegir invoke, batch o stream](#10.-Elegir-invoke,-batch-o-stream)
    - [10.1 Una entrada con `invoke`](#10.1-Una-entrada-con-invoke)
    - [10.2 Dos entradas con `batch`](#10.2-Dos-entradas-con-batch)
    - [10.3 Una respuesta por fragmentos con `stream`](#10.3-Una-respuesta-por-fragmentos-con-stream)
- [11. Combinar los pasos en una función](#11.-Combinar-los-pasos-en-una-función)
    - [11.1 Añadir un paso a la cadena](#11.1-Añadir-un-paso-a-la-cadena)
- [12. Transmitir fragmentos desde la función](#12.-Transmitir-fragmentos-desde-la-función)
- [13. Usar la variante asíncrona](#13.-Usar-la-variante-asíncrona)
- [14. Conectar componentes con el operador |](#14.-Conectar-componentes-con-el-operador-|)
    - [14.1 Añadir un paso con `|`](#14.1-Añadir-un-paso-con-|)
- [Práctica breve](#Práctica-breve)
- [Referencias](#Referencias)

**Preparación:** `uv sync --locked`, `ANTHROPIC_API_KEY` en `.env` y `uv run jupyter lab` con el kernel de `.venv`. Ver [SETUP.md](../SETUP.md).

Todo el recorrido realiza **18 llamadas a Claude Haiku**, incluidas las dos entradas de `batch`. Repetir celdas consume tokens. Las trazas de LangSmith llegan en L12.

## 0. Preparación del modelo

`ChatAnthropic` conecta LangChain con Claude. Cargar `.env` y crear el objeto no envía todavía una pregunta. Las llamadas comienzan con `invoke`, `batch`, `stream` o sus variantes asíncronas.

Usamos `claude-haiku-5-5`, rápido y económico. Este modelo no acepta `temperature` y pide indicar `thinking`: lo desactivamos porque las tareas de este notebook son cortas y no necesitan razonamiento extendido. `max_tokens` limita la longitud de cada respuesta.

In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic

# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

model = ChatAnthropic(
    model="claude-haiku-5-5",
    max_tokens=400,
    thinking={"type": "disabled"},
    timeout=60,
    max_retries=2,
)

## 1. Enviar un texto al modelo

`invoke` recibe un texto y devuelve un mensaje de respuesta. `.text` nos permite mostrar solo su texto.

In [ ]:
ticket_text = (
    "Ticket T-2048: el cliente C-1042 de ACME reporta dos cargos iguales de "
    "185.000 COP en su tarjeta débito, hechos el 12 de octubre en la misma tienda en línea."
)

response = model.invoke(f"Resume en una frase este mensaje de soporte: {ticket_text}")
print(response.text)

## 2. Enviar un mensaje de usuario

`HumanMessage` representa lo que pregunta una persona. Ahora entregamos una lista de mensajes al mismo modelo. Observa que también recibimos un mensaje de respuesta.

In [ ]:
from langchain_core.messages import HumanMessage

human_message = HumanMessage(
    content=f"{ticket_text}\n¿Qué problema reporta el cliente?"
)
response = model.invoke([human_message])
print(response.text)

## 3. Añadir instrucciones de sistema

`SystemMessage` define cómo debe responder el asistente. El mensaje humano conserva el ticket y su pregunta. Compara esta salida con la anterior: ¿se respetó el formato solicitado?

In [ ]:
from langchain_core.messages import SystemMessage

system_message = SystemMessage(
    content="Responde en español con una sola frase que comience con 'Problema:'. "
    "Usa únicamente la información del ticket."
)
response = model.invoke([system_message, human_message])
print(response.text)

## 4. Construir una plantilla de texto

`PromptTemplate` separa el texto fijo de las variables `{context}` y `{question}`. Su `invoke` **solo rellena la plantilla**: no llama al modelo. Observa el texto resultante.

In [ ]:
from langchain_core.prompts import PromptTemplate

text_template = PromptTemplate.from_template(
    "Responde brevemente en español usando solo el contexto. "
    "Si falta el dato, responde 'No se indica en el ticket'.\n\n"
    "Contexto: {context}\nPregunta: {question}"
)
values = {
    "context": ticket_text,
    "question": "¿De cuánto es cada cargo?",
}
text_prompt = text_template.invoke(values)
print(text_prompt.to_string())

## 5. Enviar la plantilla al modelo

Ya tenemos el prompt renderizado. Ahora lo pasamos al modelo. Son dos pasos distintos: preparar el mensaje y generar una respuesta.

In [ ]:
response = model.invoke(text_prompt)
print(response.text)

## 6. Construir una plantilla de mensajes

`ChatPromptTemplate` conserva los roles de sistema y usuario. Usa los mismos `values` del paso 4, pero produce otro objeto: `chat_prompt`, una lista de mensajes con roles. `text_prompt` (paso 4) era un solo texto. Esta celda tampoco llama al modelo: imprime los mensajes que enviaremos.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "Responde brevemente en español usando solo el contexto. "
               "Si falta el dato, responde 'No se indica en el ticket'."),
    ("human", "Contexto: {context}"),
    ("human", "Pregunta: {question}"),
])
chat_prompt = chat_template.invoke(values)
for message in chat_prompt.to_messages():
    print(message.type, ":", message.content)

In [ ]:
for message in chat_prompt.to_messages():
    message.pretty_print()

## 7. Enviar los mensajes al modelo

Desde aquí usamos `chat_prompt`, creado en 6. `text_prompt` cumplió su papel en 5. El modelo recibe los mensajes producidos por la plantilla. Identifica la parte que contiene las instrucciones y la que contiene nuestros datos.

In [ ]:
response = model.invoke(chat_prompt)
print(response.text)

## 8. Pedir una salida estructurada

En lugar de texto libre, pedimos dos campos: respuesta y justificación. `BaseModel` define esa forma y `with_structured_output` la aplica. Observa que podemos acceder a `answer` como atributo. La estructura no garantiza que la respuesta sea fiel al ticket: también debemos leerla.

In [ ]:
from pydantic import BaseModel, Field

class AnswerWithJustification(BaseModel):
    """Respuesta breve sobre el ticket y el dato que la sustenta."""
    answer: str = Field(description="Respuesta a la pregunta, en español.")
    justification: str = Field(description="Dato del ticket que sustenta la respuesta.")

In [ ]:
structured_model = model.with_structured_output(AnswerWithJustification)
structured_response = structured_model.invoke(chat_prompt)

In [ ]:
print(structured_response.model_dump_json(indent=2))

### 8.1 Añadir un campo

Para pedir un dato más basta con añadir un campo a la clase. Copiamos `AnswerWithJustification` con otro nombre y le agregamos `amount`. Su tipo es `int`: el modelo debe devolver un número, no un texto. En la salida, `amount` aparece sin comillas.

In [ ]:
class AnswerWithAmount(BaseModel):
    """Respuesta breve sobre el ticket, el dato que la sustenta y el monto."""
    answer: str = Field(description="Respuesta a la pregunta, en español.")
    justification: str = Field(description="Dato del ticket que sustenta la respuesta.")
    amount: int = Field(description="Monto de cada cargo en pesos colombianos, sin separadores.")  # campo nuevo

In [ ]:
amount_model = model.with_structured_output(AnswerWithAmount)
amount_response = amount_model.invoke(chat_prompt)

In [ ]:
print(amount_response.model_dump_json(indent=2))

In [ ]:
print(structured_response.answer)

In [ ]:
structured_response

## 9. Convertir texto separado por comas en una lista

Un parser transforma un formato de salida. Este ejemplo trabaja sobre un texto fijo: **no llama al modelo ni abre un archivo CSV**. La salida es una lista de Python.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
categories = parser.invoke("cargo duplicado, bloqueo de tarjeta, consulta de saldo")
print(categories)

## 10. Elegir invoke, batch o stream

Los tres métodos usan el mismo modelo. `invoke` resuelve una entrada; `batch` procesa varias entradas y devuelve las respuestas en el mismo orden; `stream` permite mostrar fragmentos a medida que llegan. Aquí `batch` hace dos llamadas y no es la API de procesamiento diferido por lotes del proveedor.

### 10.1 Una entrada con `invoke`

In [ ]:
response = model.invoke(chat_prompt)
print(response.text)

### 10.2 Dos entradas con `batch`

Cada pregunta incluye su contexto. Dos llamadas independientes no comparten una memoria de conversación.

```mermaid
sequenceDiagram
    participant N as Notebook
    participant C as Claude Haiku
    par llamada 1
        N->>C: ticket + ¿cuántos cargos?
        C-->>N: "dos cargos"
    and llamada 2
        N->>C: ticket + ¿qué tarjeta?
        C-->>N: "tarjeta débito"
    end
    Note over N: responses = [respuesta 1, respuesta 2]<br/>en el mismo orden de entrada
```

In [ ]:
batch_prompts = [
    f"{ticket_text}\n¿Cuántos cargos reporta el cliente? Responde en una frase.",
    f"{ticket_text}\n¿Con qué tarjeta se hicieron los cargos? Responde en una frase.",
]
responses = model.batch(batch_prompts)
for response in responses:
    response.pretty_print()

### 10.3 Una respuesta por fragmentos con `stream`

Pedimos una respuesta más larga para notar la diferencia con `invoke`: el texto aparece mientras se genera. El tiempo total es similar; lo que cambia es que el primer texto aparece antes. Los fragmentos pueden contener varios caracteres o venir vacíos. `end=""` y `flush=True` muestran el texto conforme llega; `time.sleep` es solo una pausa para seguirlo con la vista.

```mermaid
sequenceDiagram
    participant N as Notebook
    participant C as Claude Haiku
    N->>C: invoke(prompt)
    Note right of C: genera toda la respuesta
    C-->>N: AIMessage completo
    N->>C: stream(prompt)
    C-->>N: fragmento 1
    C-->>N: fragmento 2
    C-->>N: …
    C-->>N: último fragmento
```

In [ ]:
import time

stream_prompt = (
    f"{ticket_text}\nExplica en 5 viñetas qué datos trae este ticket y cuáles faltan "
    "para investigar el cargo (por ejemplo, el nombre de la tienda o el código de autorización). "
    "Usa solo el ticket."
)
for chunk in model.stream(stream_prompt):
    print(chunk.text, end="", flush=True)
    time.sleep(0.05)  # pausa solo para seguir los fragmentos con la vista
print()

## 11. Combinar los pasos en una función

La función muestra explícitamente cada paso: preparar el prompt y llamar al modelo. `@chain` convierte esa función en un componente con la misma interfaz `invoke`. Es una cadena de pasos; no es un agente que decide acciones.

```mermaid
flowchart LR
    V["values<br/>context + question"] --> F
    subgraph F["@chain support_assistant"]
        T["chat_template.invoke"] --> M["model.invoke"]
    end
    F --> R["AIMessage"]
```

In [ ]:
from langchain_core.runnables import chain

@chain
def support_assistant(inputs):
    prompt = chat_template.invoke(inputs)
    return model.invoke(prompt)

response = support_assistant.invoke(values)
print(response.text)

### 11.1 Añadir un paso a la cadena

Tomamos la misma función de arriba y le agregamos **un paso al final**. Agregar un paso es añadir una línea que recibe la salida del paso anterior.

**¿Qué hace `StrOutputParser`?** El modelo devuelve un `AIMessage`: el texto más metadatos como el uso de tokens. `StrOutputParser` toma ese mensaje y devuelve **solo el texto**, como un `str` de Python. Es lo mismo que hacíamos con `.text`, pero convertido en un paso de la cadena: así quien use la cadena recibe directamente el texto, listo para mostrarlo o pasarlo a otro paso. No llama al modelo. Las comillas de la salida indican que ya es texto.

```mermaid
flowchart LR
    V["values"] --> F
    subgraph F["@chain support_assistant_text"]
        T["chat_template.invoke"] --> M["model.invoke"] --> P["StrOutputParser"]
    end
    F --> R["str"]
    style P stroke-width:3px
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser

@chain
def support_assistant_text(inputs):
    prompt = chat_template.invoke(inputs)
    response = model.invoke(prompt)
    return StrOutputParser().invoke(response)  # paso nuevo

answer_text = support_assistant_text.invoke(values)
print(repr(answer_text))

## 12. Transmitir fragmentos desde la función

En 10.3 hicimos `stream` **directamente sobre el modelo**. Ahora queremos lo mismo para **nuestra propia cadena**: la función del paso 11, que primero arma el prompt y después llama al modelo.

El problema: si llamamos `support_assistant.stream(values)`, la función de 11 usa `model.invoke`, espera la respuesta completa y la entrega **en un solo fragmento**. Para que la cadena transmita, la función debe recorrer `model.stream` y entregar cada fragmento con `yield` apenas llega.

| | `model.stream` (paso 10.3) | `streaming_assistant.stream` (paso 12) |
|---|---|---|
| ¿Qué transmite? | Solo el modelo | La cadena completa: plantilla + modelo |
| ¿Quién arma el prompt? | Nosotros, antes de llamar | La función, por dentro |
| ¿Cómo se usa? | `model.stream(prompt)` | `streaming_assistant.stream(values)` |

**Beneficio:** quien use la cadena solo entrega `values` y recibe el texto mientras se genera, sin conocer la plantilla ni el modelo. Así se comporta un asistente en una interfaz de chat.

In [ ]:
@chain
def streaming_assistant(inputs):
    prompt = chat_template.invoke(inputs)
    for chunk in model.stream(prompt):
        yield chunk

for chunk in streaming_assistant.stream(values):
    print(chunk.text, end="", flush=True)
    time.sleep(0.5)  # pausa solo para seguir los fragmentos con la vista
print()

## 13. Usar la variante asíncrona

`async def`, `await` y `ainvoke` permiten esperar la respuesta sin bloquear otras tareas asíncronas. Esto no hace que una pregunta individual se responda más rápido. En Jupyter usamos `await` directamente; un script fuera del notebook usaría `asyncio.run`.

```mermaid
sequenceDiagram
    participant N as Notebook
    participant C as Claude Haiku
    participant O as Otra tarea
    N->>C: await async_assistant.ainvoke(values)
    Note over N: espera sin bloquear
    N->>O: puede atender otra tarea
    O-->>N: listo
    C-->>N: AIMessage
    Note over N: la respuesta tarda lo mismo
```

In [ ]:
@chain
async def async_assistant(inputs):
    prompt = await chat_template.ainvoke(inputs)
    return await model.ainvoke(prompt)

response = await async_assistant.ainvoke(values)
print(response.text)

## 14. Conectar componentes con el operador |

La expresión `chat_template | model` conecta la salida de la plantilla con la entrada del modelo. Hace el mismo recorrido del paso 11, escrito de forma declarativa. La cadena también ofrece `invoke` y `stream`.

```mermaid
flowchart LR
    V["values"] --> T["chat_template"] -- "|" --> M["model"] --> R["AIMessage"]
```

In [ ]:
assistant = chat_template | model

response = assistant.invoke(values)
print(response.text)

for chunk in assistant.stream(values):
    print(chunk.text, end="", flush=True)
    time.sleep(0.05)  # pausa solo para seguir los fragmentos con la vista
print()

### 14.1 Añadir un paso con `|`

Con `|` agregar un paso es añadir otro `| componente` al final. Aquí `StrOutputParser()` cumple el mismo papel que en 11.1: recibe el `AIMessage` del modelo y entrega solo su texto. Compara con la función anterior: ¿qué línea corresponde a cada `|`?

```mermaid
flowchart LR
    V["values"] --> T["chat_template"] -- "|" --> M["model"] -- "|" --> P["StrOutputParser"] --> R["str"]
    style P stroke-width:3px
```

In [ ]:
assistant_text = chat_template | model | StrOutputParser()

piped_text = assistant_text.invoke(values)
print(repr(piped_text))

## Práctica breve

1. Cambia `ticket_text` por otro mensaje sintético de un cliente de ACME, por ejemplo un bloqueo de tarjeta o una transferencia que no llegó. Vuelve a ejecutar desde el paso 1 para actualizar las variables que dependen de él.
2. En el paso 4, cambia `question` por una pregunta sobre un dato presente y ejecuta 4–7. Identifica qué variable cambia y qué plantilla se reutiliza.
3. Pregunta por un dato ausente, por ejemplo el nombre de la tienda. Comprueba si la respuesta reconoce que no está en el ticket.
4. Explica por qué 4, 6 y 9 funcionan sin llamar a Claude, y qué diferencia hay entre `support_assistant_text` (con `@chain`) y `assistant_text` (con `|`). Añade un paso más a cada una.

No escribas datos reales de clientes en los prompts: usa siempre ejemplos inventados.

## Referencias

**Learning LangChain**, Mayo Oshin y Nuno Campos, O'Reilly: [scripts del capítulo 1](https://github.com/langchain-ai/learning-langchain/tree/master/ch1/py). Se conservan el orden y los conceptos de sus 14 ejemplos, con un ticket de soporte de ACME, `ChatAnthropic` con Haiku, `.text` para mostrar las respuestas y `await` directo en Jupyter.

- [ChatAnthropic](https://docs.langchain.com/oss/python/integrations/chat/anthropic)
- [Modelos en LangChain](https://docs.langchain.com/oss/python/langchain/models)
- [Salida estructurada](https://docs.langchain.com/oss/python/langchain/structured-output)